In [ ]:
import pandas as pd
import nltk
import time
from parallel_pandas import ParallelPandas

nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.corpus import stopwords

from fakenews_functions import clean_data, tokenize_and_remove_stopwords, stemming_words

In [2]:
ParallelPandas.initialize(n_cpu=8, split_factor=4, disable_pr_bar=True)

In [3]:
sample_data = pd.read_csv("../data/news_sample.csv")
sample_content = sample_data["content"]
sample_content.dropna(inplace=True)

In [4]:
sample_content_cleaned = sample_content.apply(clean_data)
sample_content_tokens = sample_content_cleaned.apply(nltk.word_tokenize)
sample_content_nostopwords = sample_content_cleaned.apply(tokenize_and_remove_stopwords)
sample_content_stemmed = sample_content_nostopwords.apply(stemming_words)

In [5]:
counts_sample_tokens = sample_content_tokens.explode().value_counts()
counts_sample_nostopwords = sample_content_nostopwords.explode().value_counts()
counts_sample_stemmed = sample_content_stemmed.explode().value_counts()

In [6]:
print(len(counts_sample_tokens))
print(len(counts_sample_nostopwords))
print(len(counts_sample_stemmed))

16592
16446
239


In [ ]:
file_chunks = pd.read_csv("../data/995,000_rows.csv", usecols=["content"], chunksize=2**16)

counts_cleaned = pd.Series()
counts_nostopwords = pd.Series()
counts_stemmed = pd.Series()

print("Processed chunks: ", end="")
for chunk_number, chunk in enumerate(file_chunks):
    data: pd.Series = chunk["content"]
    data.dropna(inplace=True)
    
    
    data = data.p_apply(clean_data)
    tokens = data.p_apply(nltk.word_tokenize)
    counts = tokens.explode().value_counts(sort=False)
    counts_cleaned = counts_cleaned.add(counts, fill_value=0)    
    
    data = data.p_apply(tokenize_and_remove_stopwords)
    counts = data.explode().value_counts(sort=False)
    counts_nostopwords = counts_nostopwords.add(counts, fill_value=0) 
    
    data = data.p_apply(stemming_words)
    stemmed_tokens = data.p_apply(lambda s: s.split(" "))
    counts = stemmed_tokens.explode().value_counts(sort=False)
    counts_stemmed = counts_stemmed.add(counts, fill_value=0)
   
    print(f"{chunk_number} ")

Processed chunks: 0 
1 
2 
3 


In [79]:
print(len(counts_cleaned))
print(len(counts_nostopwords))
print(len(counts_stemmed))

4569899
2089905
1875210


In [8]:
print(counts_cleaned.sum())
print(counts_nostopwords.sum())
print(counts_stemmed.sum())



563110347.0
366506364.0
345689458.0


In [9]:
print(len(counts_cleaned))
print(len(counts_nostopwords))
print(len(counts_stemmed))



1955476
1955315
1742554
